# Case Study 01 — Housing Price Prediction

**Problem type:** Regression · **Dataset:** California Housing

End-to-end regression workflow: data exploration → preprocessing → baseline → advanced model → evaluation → interpretation.

In [ ]:
import sys
sys.path.append('../..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from src.evaluation import evaluate_regression, compare_models, feature_importance_df
from src.visualization import plot_correlation_heatmap, plot_residuals, plot_predicted_vs_actual, plot_feature_importance
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Problem Framing

**Goal:** Predict California median house values from neighbourhood demographic and geographic features.

**Success metric:** R² > 0.80, RMSE < $55K (vs. naive baseline)

## Step 2: Data Understanding

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
print(f'Shape: {X.shape}')
print(f'Features: {list(X.columns)}')
X.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, X.columns):
    ax.hist(X[col], bins=50, color='steelblue', alpha=0.7)
    ax.set_title(col)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 7))
corr_df = X.copy()
corr_df['target'] = y
sns.heatmap(corr_df.corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Feature Correlations (with Target)', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')

## Step 4–5: Baseline + Advanced Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    predictions[name] = preds
    results.append(evaluate_regression(y_test, preds, name=name))

compare_models(results).round(4)

## Step 6: Evaluation — Visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
best_model_name = 'Random Forest'
best_preds = predictions[best_model_name]

axes[0].scatter(y_test, best_preds, alpha=0.3, color='teal')
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'{best_model_name}: Predicted vs Actual', fontweight='bold')

residuals = y_test - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.3, color='steelblue')
axes[1].axhline(y=0, color='red', ls='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot', fontweight='bold')

plt.tight_layout()
plt.savefig('results/01_regression_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Interpretation — Feature Importance

In [ ]:
rf = models['Random Forest']
fi = feature_importance_df(X.columns.tolist(), rf.feature_importances_, top_k=8)
print(fi)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi['feature'][::-1], fi['importance'][::-1], color='teal')
ax.set_xlabel('Importance')
ax.set_title('Top Features — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('results/01_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Reflection

**Findings:** Random Forest and Gradient Boosting consistently outperform linear models, demonstrating non-linear relationships in the data (especially around income and location).

**Top predictor:** Median income (~50% feature importance) — supports the well-known correlation between household income and housing price.

**Limitations:**
- Dataset reflects 1990s California — not representative of current market
- Residuals show heteroscedasticity (variance increases with predicted value)
- No geographic spatial autocorrelation modelling

**Production considerations:**
- Update dataset annually
- Add neighbourhood-level features (crime, schools, walkability)
- Spatial cross-validation to avoid leakage between adjacent neighbourhoods

---
[← Back to main README](../../README.md)